# 03 — Extração automática por coordenada e região

Os notebooks usam somente partições mensais NetCDF processadas, presentes no disco e verificadas por SHA-256 no manifesto SQLite. Não exigem consolidação Zarr nem igual cobertura entre poluentes. Não baixam arquivos, não alteram o manifesto e não inferem observações ausentes.

Datas, variáveis e anos são descobertos a cada execução. Arquivos brutos sem processamento, caminhos quebrados, checksum incorreto, duplicidade de partições e escopos incompatíveis são reportados. Arquivos órfãos fora do manifesto não são incorporados automaticamente. A cobertura é a local: não prova completude de toda a série publicada no ADS.

Meses com qualquer célula espacial não finita são excluídos das médias espaciais para manter domínio constante. Anos completos têm 12 meses utilizáveis. Dois ou mais anos completos permitem uma **referência empírica multianual**, não necessariamente uma normal climatológica de 30 anos. Anos não consecutivos são permitidos e listados. A referência muda quando a base cresce; JSON e diretórios de execução preservam a seleção usada. Com menos de dois anos completos, há ciclo mensal descritivo e não há produto de anomalias. Gases conservam as unidades do arquivo.

A coordenada abaixo é exemplo de célula de grade, não uma estação. Não selecione datas manualmente.

In [ ]:
NOTEBOOK = "03_timeseries_example"
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import pandas as pd
from IPython.display import display, Image

# Execute da pasta do projeto ou de notebooks/. Configuração alternativa é opcional.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "config/config.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Execute o notebook dentro do projeto que contém config/config.yaml.")
sys.path.insert(0, str(ROOT / "notebooks"))
from cams_air_quality.config import load_config
from available_data import inventory, open_variable, coverage, analyze, point_and_region

CONFIG_PATH = Path(os.environ.get("CAMSAQ_CONFIG", ROOT / "config/config.yaml"))
cfg = load_config(CONFIG_PATH)
VARIABLES = None  # None: todas as variáveis mensais processadas encontradas neste escopo.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = Path(cfg["root"]) / "reports" / "automatic_notebooks" / NOTEBOOK / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
rows, file_report = inventory(cfg, VARIABLES)
file_report.to_csv(OUT / "file_inventory.csv", index=False)
variables = sorted({r["variable"] for r in rows})
print("Configuração:", CONFIG_PATH)
print("Variáveis com arquivos processados íntegros:", variables)
print("Saídas:", OUT)
if not variables:
    print("Sem dados elegíveis. Confira o inventário; execute camsaq validate e camsaq process --kind monthly.")
display(file_report)


In [ ]:
import matplotlib.pyplot as plt
LAT, LON = -23.5505, -46.6333
BBOX = None  # Exemplo: [-20, -55, -26, -40]. Independente do ponto acima.
errors = []
for variable in variables:
    try:
        point = point_and_region(variable, rows, cfg, OUT, LAT, LON, BBOX)
        table = pd.read_csv(OUT / variable / "nearest_cell.csv", parse_dates=["time"])
        display(table)
        print("Célula selecionada:", point.attrs.get("selected_latitude"),
              point.attrs.get("selected_longitude"), "distância km:", point.attrs.get("distance_km_approx"))
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(table["time"], table[variable], marker="o")
        ax.set_title(f"{variable}: célula mais próxima de ({LAT}, {LON})")
        ax.set_ylabel(point.attrs.get("units", ""))
        ax.grid(alpha=0.25)
        fig.savefig(OUT / variable / "nearest_cell.png", dpi=130, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    except Exception as exc:
        errors.append({"variable": variable, "error": str(exc)})
        print("ERRO —", variable, exc)
(OUT / "errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")
print("Falhas:", len(errors), "— tabelas e metadados:", OUT)

CSV conserva lacunas como NaN, sem interpolar. Metadados ficam nos arquivos `.metadata.json`. A extração pontual preserva meses válidos no ponto mesmo quando existem falhas em outras células; a média regional exige todos os centros do recorte finitos. Os critérios são diferentes e estão explicitados. Cada variável conserva sua própria cobertura e unidade. Não some ou compare numericamente gases em kg/kg com PM em µg/m³.